# Cadence - Stub TFLite Pipeline

## 1. Install dependencies and check notebook runtime

In [5]:
%rm /usr/local/lib/python3.12/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so

In [1]:
%pip install --upgrade torch torchao litert-torch ai-edge-litert -q

In [2]:
import importlib.util
import shutil
import subprocess
from pathlib import Path

REQUIRED_PACKAGES = [
    "torch",
    "litert_torch",
    "ai_edge_litert",
]
missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]
if missing:
    raise RuntimeError(f"Missing required packages: {missing}.")

WORK_DIR = Path("/content/cadence_stub_work")
OUTPUT_DIR = Path("/content/cadence_outputs")
OUTPUT_MODEL_PATH = OUTPUT_DIR / "stub_wav2vec2.tflite"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working files: {WORK_DIR}")
print(f"Final model: {OUTPUT_MODEL_PATH}")

if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(
        result.stdout
        if result.returncode == 0
        else "nvidia-smi is available but did not report a GPU."
    )
else:
    print("nvidia-smi not found.")

Working files: /content/cadence_stub_work
Final model: /content/cadence_outputs/stub_wav2vec2.tflite
Tue Apr 21 23:43:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                      

## 2. Create the stub model

In [3]:
import torch
import torch.nn as nn


class StubWav2Vec2(nn.Module):
    def __init__(self):
        super().__init__()
        # Downsample in stages to keep the stub small while matching the app contract.
        self.down1 = nn.Linear(320, 64)
        self.down2 = nn.Linear(64, 768)

    def forward(self, x):
        # Input: float32[1, 160000]. Output: float32[1, 500, 768].
        x = x.reshape(1, 500, 320)
        x = self.down1(x)
        return self.down2(x)


model = StubWav2Vec2().eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB")

dummy_input = torch.randn(1, 160000)
with torch.no_grad():
    output = model(dummy_input)

assert output.shape == (1, 500, 768), f"Shape mismatch: {output.shape}"
print(f"Input shape: {tuple(dummy_input.shape)}")
print(f"Output shape: {tuple(output.shape)}")

Total parameters: 70,464
Model size: ~0.27 MB
Input shape: (1, 160000)
Output shape: (1, 500, 768)


## 3. Convert directly to TFLite

In [6]:
import torchao
import litert_torch

print("torchao:", torchao.__version__)
print("litert_torch import OK")

/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:351: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(


torchao: 0.17.0
litert_torch import OK


In [7]:
sample_input = (torch.zeros(1, 160_000, dtype=torch.float32),)

print("Converting PyTorch to LiteRT/TFLite via litert_torch.")

edge_model = litert_torch.convert(model, sample_input)
edge_model.export(str(OUTPUT_MODEL_PATH))

size_mb = OUTPUT_MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"TFLite model saved: {OUTPUT_MODEL_PATH} ({size_mb:.2f} MB)")

Converting PyTorch to LiteRT/TFLite via litert_torch.
TFLite model saved: /content/cadence_outputs/stub_wav2vec2.tflite (0.27 MB)


## 4. Verify the TFLite model

In [8]:
import numpy as np
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=str(OUTPUT_MODEL_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

checks = {
    "input_name": input_details[0]["name"],
    "input_shape": tuple(input_details[0]["shape"].tolist()),
    "input_dtype": input_details[0]["dtype"],
    "output_name": output_details[0]["name"],
    "output_shape": tuple(output_details[0]["shape"].tolist()),
    "output_dtype": output_details[0]["dtype"],
}

print("Input Tensor:")
print(f"Name: {checks['input_name']}")
print(f"Shape: {checks['input_shape']}")
print(f"Type: {checks['input_dtype']}")

print("\nOutput Tensor:")
print(f"Name: {checks['output_name']}")
print(f"Shape: {checks['output_shape']}")
print(f"Type: {checks['output_dtype']}")

test_input = np.random.randn(1, 160000).astype(np.float32)
interpreter.set_tensor(input_details[0]["index"], test_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])

verification_results = [
    ("Input shape == (1, 160000)", checks["input_shape"] == (1, 160000)),
    ("Input dtype == float32", checks["input_dtype"] == np.float32),
    ("Output shape == (1, 500, 768)", checks["output_shape"] == (1, 500, 768)),
    ("Output dtype == float32", checks["output_dtype"] == np.float32),
    ("No NaN in output", not bool(np.any(np.isnan(output)))),
    ("No Inf in output", not bool(np.any(np.isinf(output)))),
]
name_results = [
    ("Input name == input_values", checks["input_name"] == "input_values"),
    ("Output name == embeddings", checks["output_name"] == "embeddings"),
]

all_passed = True
names_passed = True
print("\nTFLite Verification Results")

for label, passed in verification_results:
    print(f"- [{'PASS' if passed else 'FAIL'}] {label}")
    all_passed = all_passed and passed

for label, passed in name_results:
    print(f"- [{'PASS' if passed else 'INFO'}] {label}")
    names_passed = names_passed and passed

if not all_passed:
    raise RuntimeError(
        "Stub TFLite verification failed. Do not use this model in the app."
    )

if not names_passed:
    print("Tensor names differ from the original stub.")

print(f"\nVerified stub model: {OUTPUT_MODEL_PATH}")

Input Tensor:
Name: serving_default_args_0:0
Shape: (1, 160000)
Type: <class 'numpy.float32'>

Output Tensor:
Name: StatefulPartitionedCall:0
Shape: (1, 500, 768)
Type: <class 'numpy.float32'>

TFLite Verification Results
- [PASS] Input shape == (1, 160000)
- [PASS] Input dtype == float32
- [PASS] Output shape == (1, 500, 768)
- [PASS] Output dtype == float32
- [PASS] No NaN in output
- [PASS] No Inf in output
- [INFO] Input name == input_values
- [INFO] Output name == embeddings
Tensor names differ from the original stub.

Verified stub model: /content/cadence_outputs/stub_wav2vec2.tflite
